# Section M: Mobility & Escape Resources


**HSI Weight:** 0.15
**Effect on Model:** Mixed — higher vehicle access reduces S·Z encounters via evacuation, indirectly raising α_eff

## Sub-factors
| Variable | Source File | Key Columns |
|---|---|---|
| `vehicle_access` | `data:acs_vehicles.csv` | `B25044_001E` (total), `B25044_003E` (owner, no veh), `B25044_010E` (renter, no veh) |
| `transit_dependence` | same file | derived as % zero-car households |
| `road_density` | `data:usda_ruca.csv` | `Primary RUCA Code 2010` — lower code = more urban = higher road density |

## Output
Exports `data/nc_mobility.csv` with columns: `GEOID`, `vehicle_access`, `transit_dependence`, `road_density`, `score_M`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Ensure we are running from project root ────────────────────────────────
if os.path.basename(os.getcwd()) == 'Zombie Code':
    os.chdir('..')
print('Working directory:', os.getcwd())

VEHICLES_PATH = 'data/data:acs_vehicles.csv'
RUCA_PATH     = 'data/data:usda_ruca.csv'
OUTPUT_PATH   = 'data/nc_mobility.csv'

## 1. Vehicle Access & Transit Dependence — ACS B25044

In [ ]:
df_veh_raw = pd.read_csv(VEHICLES_PATH, dtype=str)

# Row 0 = column codes (header)
# Row 1 = human-readable labels — skip it
# B25044_001E = Total households
# B25044_003E = Owner occupied, no vehicle available
# B25044_010E = Renter occupied, no vehicle available

df_veh = (
    df_veh_raw
    .iloc[1:]  # skip label row
    [['GEO_ID', 'B25044_001E', 'B25044_003E', 'B25044_010E']]
    .assign(
        GEOID=lambda d: d['GEO_ID'].astype(str).str[-11:],
        total=lambda d: pd.to_numeric(d['B25044_001E'], errors='coerce'),
        owner_no_veh=lambda d: pd.to_numeric(d['B25044_003E'], errors='coerce').fillna(0),
        renter_no_veh=lambda d: pd.to_numeric(d['B25044_010E'], errors='coerce').fillna(0),
    )
    .dropna(subset=['total'])
    .query('total > 0')
)

# Filter to NC (FIPS prefix 37)
df_veh = df_veh[df_veh['GEOID'].str.startswith('37')].copy()

# Derive metrics
df_veh['no_veh_total']       = df_veh['owner_no_veh'] + df_veh['renter_no_veh']
df_veh['transit_dependence'] = (df_veh['no_veh_total'] / df_veh['total'] * 100).round(2)
df_veh['vehicle_access']     = (100 - df_veh['transit_dependence']).round(2)

df_veh = df_veh[['GEOID', 'vehicle_access', 'transit_dependence']].reset_index(drop=True)

print(f'Vehicle tracts: {len(df_veh):,}')
print(f'Vehicle access:      {df_veh.vehicle_access.min():.1f}% – {df_veh.vehicle_access.max():.1f}%')
print(f'Transit dependence:  {df_veh.transit_dependence.min():.1f}% – {df_veh.transit_dependence.max():.1f}%')
df_veh.head()

## 2. Road Density — USDA RUCA

In [ ]:
# RUCA file has an errata note in row 0 — skip it; real headers are in row 1
df_ruca_raw = pd.read_csv(RUCA_PATH, skiprows=1, dtype=str)

TRACT_COL = 'State-County-Tract FIPS Code (lookup by address at http://www.ffiec.gov/Geocode/)'
STATE_COL = 'Select State'
RUCA_COL  = 'Primary RUCA Code 2010'

df_ruca = (
    df_ruca_raw
    .query(f"`{STATE_COL}` == 'NC'")
    [[TRACT_COL, RUCA_COL]]
    .rename(columns={TRACT_COL: 'GEOID', RUCA_COL: 'ruca_code'})
    .assign(
        GEOID=lambda d: d['GEOID'].astype(str).str.zfill(11),
        ruca_code=lambda d: pd.to_numeric(d['ruca_code'], errors='coerce')
    )
    .dropna(subset=['ruca_code'])
    .reset_index(drop=True)
)

# ── BUG FIX: Remove RUCA code 99 (unpopulated/non-coded tracts)
# Without this filter, code 99 produces road_density = ((11-99)/10)*100 = -880.0
# which corrupts score_M for those tracts.
df_ruca = df_ruca[df_ruca['ruca_code'] <= 10]

# Convert RUCA code to road_density:
# RUCA 1 (metro core) = highest road density → score 100
# RUCA 10 (rural)     = lowest road density  → score 10
df_ruca['road_density'] = ((11 - df_ruca['ruca_code']) / 10 * 100).round(2)
df_ruca = df_ruca[['GEOID', 'road_density']]

print(f'RUCA tracts (after removing code 99): {len(df_ruca):,}')
print(f'Road density range: {df_ruca.road_density.min():.1f} – {df_ruca.road_density.max():.1f}')
print(f'Expected range: 10.0 – 100.0 (no negatives)')
assert df_ruca.road_density.min() >= 0, 'BUG: negative road density still present!'
print('✓ No negative road_density values — fix confirmed.')
df_ruca.head()


## 3. Merge All M Sub-factors

In [ ]:
df_M = df_veh.merge(df_ruca, on='GEOID', how='outer')

print(f'Merged M dataframe: {df_M.shape}')
print(f'Missing values:\n{df_M.isnull().sum()}')
df_M.head()

## 4. Normalize & Score

In [ ]:
def normalize(series: pd.Series) -> pd.Series:
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# Fill missing with median
for col in ['vehicle_access', 'transit_dependence', 'road_density']:
    df_M[col] = df_M[col].fillna(df_M[col].median())

df_M['veh_norm']     = normalize(df_M['vehicle_access'])      # positive
df_M['road_norm']    = normalize(df_M['road_density'])         # positive
df_M['transit_norm'] = normalize(df_M['transit_dependence'])   # negative — flip

# score_M: 1 = most mobile, 0 = least mobile
df_M['score_M'] = (
    df_M['veh_norm']           * (1/3) +
    df_M['road_norm']          * (1/3) +
    (1 - df_M['transit_norm']) * (1/3)
)

print(f'score_M range: {df_M.score_M.min():.3f} – {df_M.score_M.max():.3f}')
print(f'score_M mean:  {df_M.score_M.mean():.3f}')
df_M[['GEOID','vehicle_access','transit_dependence','road_density','score_M']].head(10)

## 5. Visualize

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Section M — Mobility & Escape Resources (NC Census Tracts)', fontsize=13, fontweight='bold')

axes[0].hist(df_M['vehicle_access'], bins=40, color='#3498db', edgecolor='white', linewidth=0.3)
axes[0].set_title('Vehicle Access (%)')
axes[0].set_xlabel('% Households with Vehicle')

axes[1].hist(df_M['transit_dependence'], bins=40, color='#e74c3c', edgecolor='white', linewidth=0.3)
axes[1].set_title('Transit Dependence (%)')
axes[1].set_xlabel('% Zero-car Households')

axes[2].hist(df_M['road_density'], bins=20, color='#1abc9c', edgecolor='white', linewidth=0.3)
axes[2].set_title('Road Density\n(RUCA proxy, 0–100)')
axes[2].set_xlabel('Score')

axes[3].hist(df_M['score_M'], bins=40, color='#f39c12', edgecolor='white', linewidth=0.3)
axes[3].set_title('score_M (composite)')
axes[3].set_xlabel('0 = least mobile, 1 = most')
axes[3].axvline(df_M['score_M'].mean(), color='red', linestyle='--',
                label=f'mean={df_M["score_M"].mean():.2f}')
axes[3].legend()

plt.tight_layout()
plt.savefig('data/fig_M_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- KEY FINDINGS ---')
high_transit = df_M[df_M['transit_dependence'] > 20]
print(f'Tracts >20% zero-car households (urban entrapment risk): {len(high_transit)}')
print('\nTop 5 most mobile tracts:')
print(df_M.nlargest(5, 'score_M')[['GEOID','vehicle_access','transit_dependence','road_density','score_M']].to_string(index=False))
print('\nTop 5 least mobile tracts (highest zombie entrapment risk):')
print(df_M.nsmallest(5, 'score_M')[['GEOID','vehicle_access','transit_dependence','road_density','score_M']].to_string(index=False))

## 6. Export

In [ ]:
df_out = df_M[['GEOID','vehicle_access','transit_dependence','road_density','score_M']].copy()
df_out.to_csv(OUTPUT_PATH, index=False)
print(f'Exported {len(df_out):,} tracts to {OUTPUT_PATH}')
df_out.describe()